# indy_mech_extension — hidden-state extraction for persona / broken-text probes
Teacher-forces the 528 stored Qwen3-8B rollouts from phase 17 (`phase17_qwen_wide_surveys.json`) under their own
triggers in a single forward pass each, and stores the residual stream at every 2nd layer (0..36) at 10 positions.
Rig is `phase17/hbar-rerun/hbar_personas.ipynb` cell 1, unchanged. See `PLAN.md`.

In [ ]:
# === CELL 1 — rig, identical to phase 17 / hbar-rerun ==========================================
import torch, torch.nn.functional as F, math, json, time, inspect, statistics, unicodedata, os
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
LN2 = math.log(2)
Q   = "what shall i do today"

def save(name, obj):
    with open("/content/" + name, "w") as f: json.dump(obj, f, ensure_ascii=False)
    print("  saved", name)

def load(model_id, token=None):
    global model, tokenizer, dev, V, EOS, CEIL, _LTK, TOKSTR, MODEL_ID
    global CLEAN_STR, CLEAN, PRE_P, SUF_P
    MODEL_ID = model_id
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=DTYPE, device_map="cuda:0", token=token)
    model.eval(); model.requires_grad_(False)
    dev, V, EOS = model.device, model.config.vocab_size, tokenizer.eos_token_id
    CEIL = math.log2(V)
    _LTK = ("logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters
            else "num_logits_to_keep")
    TOKSTR = tokenizer.batch_decode([[i] for i in range(V)])
    _build_scaffold()
    print(f"{model_id} | vocab {V} | ceiling {CEIL:.3f} bits | last-logit kwarg {_LTK}")

def _chat(text):
    try:
        return tokenizer.apply_chat_template([{"role":"user","content":text}], tokenize=False,
                                             add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template([{"role":"user","content":text}], tokenize=False,
                                             add_generation_prompt=True)

def _build_scaffold():
    global CLEAN_STR, CLEAN, PRE_P, SUF_P
    CLEAN_STR = _chat(Q)
    enc = tokenizer(CLEAN_STR, add_special_tokens=False, return_offsets_mapping=True)
    CLEAN, offs = enc["input_ids"], enc["offset_mapping"]
    i0 = CLEAN_STR.index(Q)
    j0 = next(t for t,(a,b) in enumerate(offs) if b > i0)
    PRE_P, SUF_P = CLEAN[:j0], CLEAN[j0:]
    assert PRE_P + SUF_P == CLEAN, "scaffold split does not round-trip"
    print(f"  clean {len(CLEAN)} tok | prefix split {len(PRE_P)}+{len(SUF_P)} (round-trip OK)")

def pre_ids(trig): return list(PRE_P) + list(trig) + list(SUF_P)

@torch.no_grad()
def gen(ids, n_new=96, seed=None):
    x = torch.tensor([ids], device=dev); out=[]; past=None
    if seed is not None: torch.manual_seed(seed)
    for _ in range(n_new):
        o = model(x, past_key_values=past, use_cache=True, **{_LTK: 1})
        past = o.past_key_values
        lg = o.logits[0,-1].float()
        nx = int(torch.multinomial(lg.softmax(-1), 1))
        if nx == EOS: break
        out.append(nx); x = torch.tensor([[nx]], device=dev)
    return out

@torch.no_grad()
def H1_of(ids):
    lg = model(torch.tensor([ids], device=dev), **{_LTK:1}).logits[0,-1].float()
    lp = F.log_softmax(lg,-1)
    return float(-(lp.exp()*lp).sum()/LN2)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print("userdata unavailable:", type(e).__name__)
os.environ["HF_TOKEN"] = HF_TOKEN or ""
import transformers; print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("rig ready")


In [ ]:
# === CELL 2 — load Qwen3-8B and reproduce phase 17's clean H1 (0.237) as the port check ========
load("Qwen/Qwen3-8B", token=HF_TOKEN or None)
h_clean = H1_of(CLEAN)
print(f"clean H1 = {h_clean:.4f} bits  (phase 17: 0.237)")
assert abs(h_clean - 0.237) < 0.01, "rig does not reproduce phase 17 — stop"
print("model config: layers", model.config.num_hidden_layers, "| hidden", model.config.hidden_size)


In [ ]:
# === CELL 3 — rebuild the exact response ids for all 528 rollouts (retok, else regen from seed) ==
SRC = json.load(open("/content/phase17_qwen_wide_surveys.json"))
ROLL = []   # one entry per rollout, in a fixed order
t0 = time.time(); n_retok = n_regen = n_fail = 0
for arm_name, arm in SRC["arms"].items():
    trig = arm.get("ids")
    prompt = CLEAN if not trig else pre_ids(trig)
    for r in arm["rollouts"]:
        want_first, want_n = r["first_id"], r["n"]
        ans = tokenizer.encode(r["text"], add_special_tokens=False)
        ok = bool(ans) and ans[0] == want_first and len(ans) == want_n
        how = "retok"
        if not ok:
            ans = gen(prompt, 96, seed=r["seed"])
            ok = bool(ans) and ans[0] == want_first and len(ans) == want_n
            how = "regen"; n_regen += 1
        if not ok:
            n_fail += 1; how = "FAIL"
            print("  FAIL", arm_name, r["seed"], "first", ans[:1], "want", want_first, "n", len(ans), "want", want_n)
        else:
            if how == "retok": n_retok += 1
        # for regen, also check the decoded text matches the stored text
        if ok and how == "regen":
            txt = tokenizer.decode(ans, skip_special_tokens=True)
            if txt != r["text"]: print("  regen text differs:", arm_name, r["seed"])
        ROLL.append(dict(arm=arm_name, seed=r["seed"], prompt_ids=prompt, resp_ids=ans if ok else None,
                         how=how, n=len(ans), first_id=want_first))
print(f"retok {n_retok} | regen {n_regen} | fail {n_fail} | {time.time()-t0:.0f}s | total {len(ROLL)}")
assert n_fail == 0, "some rollouts could not be rebuilt — do not extract"
save("rollout_ids.json", [dict(arm=x["arm"], seed=x["seed"], how=x["how"], n=x["n"],
                               prompt_ids=x["prompt_ids"], resp_ids=x["resp_ids"]) for x in ROLL])


In [ ]:
# === CELL 4 — one teacher-forced forward pass per rollout, store residual stream ================
import numpy as np
NL = model.config.num_hidden_layers          # 36
LAYERS = list(range(0, NL + 1, 2))           # 0 = embeddings, 36 = final norm input... see note
D = model.config.hidden_size
KS = [1, 2, 4, 8, 16, 32, 64]
SLOTS = ["P"] + [f"R{k}" for k in KS] + ["Rmean", "Rlast"]
# NOTE: hidden_states[i] is the residual stream AFTER block i (hidden_states[0] = embeddings);
# hidden_states[36] has the final RMSNorm applied by HF for Qwen3 — we keep it as "layer 36".
X = np.full((len(ROLL), len(LAYERS), len(SLOTS), D), np.nan, dtype=np.float16)
t0 = time.time()
with torch.no_grad():
    for i, r in enumerate(ROLL):
        p, a = r["prompt_ids"], r["resp_ids"]
        ids = torch.tensor([list(p) + list(a)], device=dev)
        out = model(ids, output_hidden_states=True, **{_LTK: 1})
        hs = torch.stack([out.hidden_states[l][0] for l in LAYERS])   # [L, T, D]
        P0 = len(p) - 1; n = len(a)
        # position P0+j is the state that has seen response tokens 1..j (j=0 -> prompt only)
        feats = [hs[:, P0]]                                           # P
        for k in KS:
            feats.append(hs[:, P0 + k] if n >= k else torch.full_like(hs[:, P0], float("nan")))
        feats.append(hs[:, P0 + 1 : P0 + n + 1].float().mean(1))      # Rmean over response positions
        feats.append(hs[:, P0 + n])                                   # Rlast
        X[i] = torch.stack(feats, 1).float().cpu().numpy().astype(np.float16)
        if i % 50 == 0: print(f"  {i}/{len(ROLL)}  {time.time()-t0:.0f}s")
print(f"done {time.time()-t0:.0f}s | X {X.shape} | nan slots {np.isnan(X[:,0,:,0]).sum()}")
meta = dict(model=MODEL_ID, layers=LAYERS, slots=SLOTS, ks=KS, hidden=D,
            rollouts=[dict(arm=x["arm"], seed=x["seed"], how=x["how"], n=x["n"]) for x in ROLL])
np.savez("/content/features_qwen_wide.npz", X=X)
save("features_meta.json", meta)
print("size MB", os.path.getsize("/content/features_qwen_wide.npz")/1e6)


In [ ]:
# === CELL 5 — quick in-Colab sanity probe so nothing is wasted if the download is slow =========
# mass-mean, leave-one-arm-out, on the crude 2-judge label carried in the survey file is NOT
# available here; instead just check that arms are separable at slot P (they must be: P is
# identical within an arm) and that Rmean differs between NULL arms and trigger arms.
import numpy as np
arms = np.array([x["arm"] for x in ROLL])
li = LAYERS.index(20); sP = SLOTS.index("P"); sM = SLOTS.index("Rmean")
XP = X[:, li, sP].astype(np.float32); XM = X[:, li, sM].astype(np.float32)
print("within-arm P spread (should be ~0):", float(np.mean([XP[arms==a].std(0).mean() for a in set(arms)])))
isnull = np.array([a.startswith("NULL") for a in arms])
d = XM[isnull].mean(0) - XM[~isnull].mean(0)
proj = XM @ d
from sklearn.metrics import roc_auc_score
print("null-vs-trigger AUROC at layer 20 Rmean (mass-mean, in-sample):", roc_auc_score(isnull, proj))
